In [2]:
# General imports
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from qiskit_ibm_runtime import QiskitRuntimeService
 
# After using the following code, go to the IBM Cloud API keys page (https://cloud.ibm.com/iam/apikeys)
# and disable your API key (click Enabled slider in the API key field)
service = QiskitRuntimeService(channel="ibm_cloud", 
                               token="give_token",
                               instance="give_instance ")

# Or save your credentials on disk.
# QiskitRuntimeService.save_account(channel='ibm_quantum', instance='ibm-q-hub-ntu/chen-jiunn-wei/default', token='<IBM Quantum API key>')
backend = service.backend("ibm_boston")
#backend = service.backend("ibm_aachen")
#backend = service.least_busy(operational=True, simulator=False)
backend.name

'ibm_boston'

In [2]:
from dataclasses import asdict
from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

#Initialized Sampler
sampler = Sampler(mode=backend, options={"default_shots": 4000})
#sampler = Sampler(mode=backend)
#sampler.options.resilience_level = 0
sampler.options.dynamical_decoupling.enable = True
sampler.options.twirling.enable_gates = True
sampler.options.twirling.enable_measure = True
sampler.options.twirling.num_randomizations = 100
sampler.options.update(default_shots=4000, dynamical_decoupling={"sequence_type": "XY4"})
#sampler.options.update(dynamical_decoupling={"sequence_type": "XY4"})

print(asdict(sampler.options))
print(backend)

{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'callback': None, 'job_tags': None, 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_shots': 4000, 'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4', 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'execution': {'init_qubits': Unset, 'rep_delay': Unset, 'meas_type': Unset}, 'twirling': {'enable_gates': True, 'enable_measure': True, 'num_randomizations': 100, 'shots_per_randomization': Unset, 'strategy': Unset}, 'experimental': Unset}
<IBMBackend('ibm_boston')>


In [3]:
# =============================================================================
# Rényi-2 Entropy Calculation for SU(2) Lattice Gauge Theory Using Qiskit
# =============================================================================
#
# Description:
# This code implements the quantum circuits and measurement operators required
# to study the real-time evolution of a minimally truncated SU(2) lattice
# gauge theory mapped onto a one-dimensional spin-1/2 chain.
#
# The main objective is to calculate the second-order Rényi entropy,
#
#                  S_A^(2) = -ln[Tr(rho_A^2)],
#
# where rho_A is the reduced density matrix of a subsystem A.
#
# Hamiltonian:
# The effective spin Hamiltonian contains the following Pauli interactions:
#
#       H = Hzz + Hz + Hx + Hzx + Hxz + Hzxz
#
# where:
#   Hzz  : Nearest-neighbor ZZ interactions
#   Hz   : Single-qubit Z interactions
#   Hx   : Single-qubit X interactions
#   Hzx  : Two-qubit ZX interactions
#   Hxz  : Two-qubit XZ interactions
#   Hzxz : Three-qubit ZXZ interactions
#
# Code Structure:
# 1. Define custom quantum gates for implementing Hamiltonian evolution.
# 2. Implement basis rotations for measuring Pauli expectation values.
# 3. Construct measurement circuits without using the Hadamard test.
# 4. Set physical parameters and system size.
# 5. Generate all 4^3 = 64 three-qubit Pauli operators for reconstructing
#    the reduced density matrix of a three-qubit subsystem.
#
# Method:
# The reduced density matrix is reconstructed from measured Pauli
# expectation values using
#
#       rho_A = (1/8) * sum_P <P> P,
#
# where P belongs to {I, X, Y, Z}^{⊗3}.
#
# The subsystem purity and Rényi-2 entropy are then calculated as
#
#       Purity = Tr(rho_A^2)
#       S_A^(2) = -ln ( (1/8 * sum_P <P>^2))
#       or
#       S_A^(2) = -ln(Purity)
#
# Software:
#   Python, Qiskit, Qiskit Aer, Qiskit IBM Runtime, NumPy
#
# =============================================================================
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator 
from qiskit import QuantumCircuit, transpile
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.quantum_info import PauliList
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import itertools
import numpy as np

#Total Shots
shots = 4000

# -------------------------------------------------------------
# 1. Custom Gates
# Gates necessary for time evolution : e^{-i(Hzz + Hz + Hx + Hzx + Hxz + Hzxz) delta_t}
# -------------------------------------------------------------
def Hzz_gate(theta):
    qc = QuantumCircuit(2)
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    return qc.to_gate(label="Hzz")

def Hz_gate(theta):
    qc = QuantumCircuit(1)
    qc.rz(theta, 0)
    return qc.to_gate(label="Hz")

def Hx_gate(theta):
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.rz(theta, 0)
    qc.h(0)
    return qc.to_gate(label="Hx")

def Hzx_gate(theta):
    qc = QuantumCircuit(2)
    qc.h(1)
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(1)
    return qc.to_gate(label="Hzx")

def Hxz_gate(theta):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(0)
    return qc.to_gate(label="Hxz")

def Hzxz_gate(theta):
    qc = QuantumCircuit(3)
    qc.h(1)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.rz(theta, 2)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(1)
    return qc.to_gate(label="Hzxz")
#-------------------------------------------------------------
# 1A. Change of Basis for calculation of expectation value of Pauli gates  <\psi(t)| Pauli_op |psi(t) >
# Pauli_op = X, Y , Z or I
#--------------------------------------------------------------
def change_basis(qc, P, target):
    if P == "X":
        qc.h(target)
        
    elif P == "Y":
        qc.sdg(target)
        qc.h(target)
        
    elif P == "Z":
        pass
        
    elif P == "I":
        pass
#        #qc.barrier()
# -------------------------------------------------------------
# 2. No Hadamard Test Construction for <P>_ψ
# -------------------------------------------------------------

def measure_pauli_expectation(base_circuit, P1, P2, P3, q1, q2, q3):
    """
    Return expectation value <\psi (t)|P1 P2 P3|\psi(t)> on 'target q1, q2, q3' qubit of state prepared by state_prep_circ,
    measured with 'shots' by rotating into Z basis and measuring.
    """

    L = base_circuit.num_qubits
    qc = QuantumCircuit(L,3)   
    
    # Load state |psi> into qubits 1..L
    qc.compose(base_circuit, inplace=True)

    t1 = q1
    t2 = q2
    t3 = q3

    # rotate to Z basis
    if P1 != "I":
        change_basis(qc, P1, t1)
        qc.measure(t1, 0)

    if P2 != "I":
        change_basis(qc, P2, t2)
        qc.measure(t2, 1)

    if P3 != "I":
        change_basis(qc, P3, t3)
        qc.measure(t3, 2)
        
    qc.barrier()
    return qc
# -------------------------------------------------------------
# 3. Physics Parameters
# -------------------------------------------------------------
a = 1.0                # lattice spacing
g = np.sqrt(1.2/a)     # gauge coupling

J = -3*(g**2)/16
hz = -2*J
hx = -2/((a*g)**2)

# system size
L = 101
print(f"L= {L}")

# 3-qubit Pauli basis (64 operators)
Psingle = ["I", "X", "Y", "Z"]
Pauli_combination = [(p1, p2, p3) for p1 in Psingle for p2 in Psingle for p3 in Psingle]

# convert the Pauli_string_op  (p1p2p3) into matrix form
matrices = [SparsePauliOp.from_list([(p1+p2+p3, 1)]).to_matrix() for (p1,p2,p3) in Pauli_combination]
print(Pauli_combination)


L= 101
[('I', 'I', 'I'), ('I', 'I', 'X'), ('I', 'I', 'Y'), ('I', 'I', 'Z'), ('I', 'X', 'I'), ('I', 'X', 'X'), ('I', 'X', 'Y'), ('I', 'X', 'Z'), ('I', 'Y', 'I'), ('I', 'Y', 'X'), ('I', 'Y', 'Y'), ('I', 'Y', 'Z'), ('I', 'Z', 'I'), ('I', 'Z', 'X'), ('I', 'Z', 'Y'), ('I', 'Z', 'Z'), ('X', 'I', 'I'), ('X', 'I', 'X'), ('X', 'I', 'Y'), ('X', 'I', 'Z'), ('X', 'X', 'I'), ('X', 'X', 'X'), ('X', 'X', 'Y'), ('X', 'X', 'Z'), ('X', 'Y', 'I'), ('X', 'Y', 'X'), ('X', 'Y', 'Y'), ('X', 'Y', 'Z'), ('X', 'Z', 'I'), ('X', 'Z', 'X'), ('X', 'Z', 'Y'), ('X', 'Z', 'Z'), ('Y', 'I', 'I'), ('Y', 'I', 'X'), ('Y', 'I', 'Y'), ('Y', 'I', 'Z'), ('Y', 'X', 'I'), ('Y', 'X', 'X'), ('Y', 'X', 'Y'), ('Y', 'X', 'Z'), ('Y', 'Y', 'I'), ('Y', 'Y', 'X'), ('Y', 'Y', 'Y'), ('Y', 'Y', 'Z'), ('Y', 'Z', 'I'), ('Y', 'Z', 'X'), ('Y', 'Z', 'Y'), ('Y', 'Z', 'Z'), ('Z', 'I', 'I'), ('Z', 'I', 'X'), ('Z', 'I', 'Y'), ('Z', 'I', 'Z'), ('Z', 'X', 'I'), ('Z', 'X', 'X'), ('Z', 'X', 'Y'), ('Z', 'X', 'Z'), ('Z', 'Y', 'I'), ('Z', 'Y', 'X'), ('Z', 

In [4]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator 
from qiskit import QuantumCircuit, transpile
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.quantum_info import PauliList
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import itertools
import numpy as np

# -------------------------------------------------------------
# 4. Create main evolution circuit
# delta_t = step size of the time trotterization 
# -------------------------------------------------------------
def quant_circuit(qc_tot, delta_t):
    theta_1 = J*2*delta_t
    theta_2 = hz*2*delta_t
    theta_3 = (hx/16)*2*delta_t
    theta_4 = -(3*hx/16)*2*delta_t
    theta_5 = -(3*hx/16)*2*delta_t
    theta_6 = (9*hx/16)*2*delta_t
    # 1st Evolution zz gate
    gate_zz  = Hzz_gate(theta_1)
    gate_z   = Hz_gate(theta_2)
    gate_x   = Hx_gate(theta_3)
    gate_zx  = Hzx_gate(theta_4)
    gate_xz  = Hxz_gate(theta_5)
    gate_zxz = Hzxz_gate(theta_6)
    
    for i in range(0,L-1,2):
        qc_tot.append(gate_zz, [i,i+1])
    for i in range(1,L,2):
        qc_tot.append(gate_zz, [i,i+1])
    #qc_tot.barrier()
    for i in range(L):
        qc_tot.append(gate_z, [i])
    for i in range(L):
        qc_tot.append(gate_x, [i])
    for i in range(0,L-1,2):
        qc_tot.append(gate_zx, [i,i+1])
    for i in range(1,L-1,2):
        qc_tot.append(gate_zx, [i,i+1])
    for i in range(0,L-1,2):
        qc_tot.append(gate_xz, [i,i+1])
    for i in range(1,L-1,2):
        qc_tot.append(gate_xz, [i,i+1])
    for i in range(0,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    for i in range(1,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    for i in range(2,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    #qc_tot.barrier()
    return qc_tot

#############################################################################
# Expextation value calulated for a Pauli string of 3 operator
def pauli_value_from_bits(P1, P2, P3, bits):
    b0 = bits[2]  # 0th bit from bits measurement
    b1 = bits[1]  # 1st bit from bits measurement
    b2 = bits[0]  # 2nd bit from bits measurement

    # Case 1: I ⊗ I ⊗ I
    if P1 == "I" and P2 == "I" and P3 == "I":
        value = 1.0
        return value
        
    # Case 2: I ⊗ I ⊗ P3
    if P1 == "I" and P2 == "I":
        value = (-1)**b2
        return  value      # only second qubit matters
    # Case 3: I ⊗ P2 ⊗ I
    elif P1 == "I" and P3 == "I":
        value = (-1)**b1
        return  value      # only first qubit matters
        
    # Case 4: P1 ⊗ I ⊗ I
    elif P2 == "I" and P3 == "I":
        value = (-1)**b0 
        return value      # only first qubit matters

    # Case 5: I ⊗ P2 ⊗ P3
    elif P1 == "I":
        value = (-1)**b1 * (-1)**b2 
        return value

    # Case 6: P1 ⊗ I ⊗ P3
    elif P2 == "I":
        value = (-1)**b0 * (-1)**b2
        return value

    # Case 7: P1 ⊗ P2 ⊗ I
    elif P3 == "I":
        value = (-1)**b0 * (-1)**b1
        return value

    # Case 8: P1 ⊗ P2 ⊗ P3
    else:
        value = (-1)**b0 * (-1)**b1 * (-1)**b2
        return value
#######################################################################
# List of number of trotter step
Num_step_list = np.array([3])

initial_t = 0.0
final_t = 1.0

total_t = 0

times = []
entropies = []
bitstrings_array_time = []
exp_array_time = []
# -------------------------------------------------------------
# 5. Main Time Evolution Loop
# -------------------------------------------------------------
for Num_step in Num_step_list:
    total_t = 0
    delta_tV2 = 0.0
    # -------- Compute entropy at t = 0 (before evolution) -------
    times.append(delta_tV2)
    # Beginning of Quantum Circuit  which is all spin down for t =0
    qc_tot_step1 = QuantumCircuit(L)
    for i in range(L):
        qc_tot_step1.x(i)

    # Add Quantum Circuit with qc_tot_step1.
    qc_tot_step1 = quant_circuit(qc_tot_step1, delta_tV2)

    # target = qubit index whose entropy we measure
    qA = (L//2)-1
    qB = L//2
    qC = (L//2)+1
    bitstrings_list = []    # it collect the bitstring data for all Pauli_string_ops  at t=0
    exp_list = []           # it collect expectaion value of all Pauli_string_ops    at t=0
    for P1,P2,P3 in Pauli_combination:
        if P1 == "I" and P2 == "I" and P3 == "I":
            exp = 1
            exp_list.append(exp)
            # number of shots entries, each entry is the bitstring "00"
            bits = ["000"] * shots
            bitstrings_list.append(bits)
        else:
            circ_R = measure_pauli_expectation(qc_tot_step1, P1, P2, P3, qA, qB, qC)
            transpiled_qc = transpile(circ_R)
            pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
            isa_circ_R = pm.run(transpiled_qc)

            job_R = sampler.run([isa_circ_R])
            counts_R = job_R.result()[0].data.c.get_counts()
            bitstrings_R = job_R.result()[0].data.c.get_bitstrings()
            bitstrings_list.append(bitstrings_R)

            exp_V2 = 0
            for bitstr, c in counts_R.items():
                b = [int(q) for q in bitstr]

                value = pauli_value_from_bits(P1, P2, P3, b)
                
                exp_V2 = exp_V2 + value *(c / shots)

            exp_list.append(exp_V2)
        
    # add bitstrings for all operator with real and Imag measurment to the bitstrings_array_time
    bitstrings_array_time.append(bitstrings_list)

    # Build ρ = (1/2**3) Σ exp[P1 P2 P3] * P1*P2*P3
    rho = np.zeros((2**3,2**3), dtype=complex)
    for exp, M in zip(exp_list, matrices):
        rho += (1/(2**3)) * exp * M
        
    #rho = rho / np.trace(rho)          

    # Entanglement entropy
    vals = np.linalg.eigvals(rho)
    vals = np.real(np.clip(vals,0,1))
    vals = np.real(vals[vals > 1e-12])
    #print("eigvals", vals)
    S_A = -np.sum(vals * np.log(vals))    # This calculate Entanglement entropy S_A

    # If you want to calculate the Renyi-2 entropy S_A2  = -ln(Tr(rho^2)) then impliment the following 2 line
    #R2 = np.sum((np.real(vals))**2)
    #S_A = - np.log(R2)
    
    print(f"EE at time ={delta_tV2} : {S_A:.4f}")

    entropies.append(S_A)

    time_step = (final_t-initial_t)/Num_step

    # Begining of Quantum Circuit  which is all spin down for t =/ 0
    qc_tot = QuantumCircuit(L)
    for i in range(L):
        qc_tot.x(i)
    # -------------------------------------------------------------
    # 6. Calculation of Entropy for t= 1/3, 2/3, and 1
    # -------------------------------------------------------------
    for i in range(Num_step):
        delta_t = time_step
        total_t = total_t + delta_t
        times.append(total_t)
        # Add Quantum Circuit with qc_tot
        qc_tot = quant_circuit(qc_tot, delta_t)

        bitstrings_list = []    # it collect the bitstring data for all Pauli_string_ops  at t=/ 0
        exp_list = []           # it collect expectaion value of all Pauli_string_ops    at t=/ 0
        for P1,P2,P3 in Pauli_combination:
            if P1 == "I" and P2 == "I" and P3 == "I":
                exp =1
                exp_list.append(exp)
                # number of shots entries, each entry is the bitstring "00"
                bits = ["000"] * shots
                bitstrings_list.append(bits)
            else:
                circ_R = measure_pauli_expectation(qc_tot, P1, P2, P3, qA, qB, qC)
                transpiled_qc = transpile(circ_R)
                pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
                isa_circ_R = pm.run(transpiled_qc)

                job_R = sampler.run([isa_circ_R])
                counts_R = job_R.result()[0].data.c.get_counts()
                bitstrings_R = job_R.result()[0].data.c.get_bitstrings()
                bitstrings_list.append(bitstrings_R)

                exp_V2 = 0
                for bitstr, c in counts_R.items():
                    b = [int(q) for q in bitstr]

                    value = pauli_value_from_bits(P1, P2, P3, b)
                
                    exp_V2 = exp_V2 + value *(c / shots)

                exp_list.append(exp_V2)

        # add bitstrings for all operator with real and Imag measurment to the bitstrings_array_time
        bitstrings_array_time.append(bitstrings_list)

        # Build ρ = (1/2**3) Σ exp[P1 P2 P3] * P1*P2*P3
        rho = np.zeros((2**3, 2**3), dtype=complex)
        for exp, M in zip(exp_list, matrices):
            rho += (1/(2**3)) * exp * M

        #rho = rho / np.trace(rho)

        # Entanglement entropy
        vals = np.linalg.eigvals(rho)
        vals = np.real(np.clip(vals,0,1))
        vals = np.real(vals[vals > 1e-12])
        #print("eigvals", vals)
        S_A = -np.sum(vals * np.log(vals))

        # If you want to calculate the Renyi-2 entropy S_A2  = -ln(Tr(rho^2)) then impliment the following 2 line
        #R2 = np.sum((np.real(vals))**2)
        #S_A = - np.log(R2)
        
        print(f"Time = {total_t:.3f}")
        print(f"Entanglement entropy : {S_A:.4f}")

        entropies.append(S_A)

print(f"times = {times}")
print(f"entropy = {entropies}")
# -------------------------------------------------------------
# 7. Save results
# -------------------------------------------------------------
with open("bitstring_arrays_EE_vs_time_N101_A3_REAL_QC_2ndRun.txt", "w") as file:
    for t, bitstrings_list in zip(times, bitstrings_array_time):
        file.write(f"time={t}:\n")
        for idx,bitstring in zip(Pauli_combination, bitstrings_list):
            file.write(f" Real part of Op{idx} = {bitstring}\n")
        file.write("\n")  # blank line for readability

np.savetxt("entropy_vs_time_hadamard_sampler_N101_A3_REAL_QC_2ndRun.txt", np.column_stack([times, entropies]),
           header="time  S_A", fmt="%.6f ")
#circ_R.draw("mpl")

EE at time =0.0 : 0.2153
Time = 0.333
Entanglement entropy : 1.2014
Time = 0.667
Entanglement entropy : 1.6215
Time = 1.000
Entanglement entropy : 1.7934
times = [0.0, np.float64(0.3333333333333333), np.float64(0.6666666666666666), np.float64(1.0)]
entropy = [np.float64(0.2153423583803638), np.float64(1.2014343710485198), np.float64(1.6215051196909733), np.float64(1.7934179123870804)]


In [5]:
# 3-qubit Pauli basis contains only I and Z (8 operators)
Psingle = ["I", "Z"]
Pauli_combination = [(p1, p2, p3) for p1 in Psingle for p2 in Psingle for p3 in Psingle]

#matrices = [SparsePauliOp.from_list([(p1+p2+p3, 1)]).to_matrix() for (p1,p2,p3) in Pauli_combination]
print(Pauli_combination)

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator 
from qiskit import QuantumCircuit, transpile
#from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.quantum_info import PauliList
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import itertools
import numpy as np

# -------------------------------------------------------------
# 4. Create main evolution circuit
# -------------------------------------------------------------
def quant_circuit(qc_tot, delta_t):
    theta_1 = J*2*delta_t
    theta_2 = hz*2*delta_t
    theta_3 = (hx/16)*2*delta_t
    theta_4 = -(3*hx/16)*2*delta_t
    theta_5 = -(3*hx/16)*2*delta_t
    theta_6 = (9*hx/16)*2*delta_t
    # 1st Evolution zz gate
    gate_zz  = Hzz_gate(theta_1)
    gate_z   = Hz_gate(theta_2)
    gate_x   = Hx_gate(theta_3)
    gate_zx  = Hzx_gate(theta_4)
    gate_xz  = Hxz_gate(theta_5)
    gate_zxz = Hzxz_gate(theta_6)
    
    for i in range(0,L-1,2):
        qc_tot.append(gate_zz, [i,i+1])
    for i in range(1,L,2):
        qc_tot.append(gate_zz, [i,i+1])
    qc_tot.barrier()
    for i in range(L):
        qc_tot.append(gate_z, [i])
    for i in range(L):
        qc_tot.append(gate_x, [i])
    for i in range(0,L-1,2):
        qc_tot.append(gate_zx, [i,i+1])
    for i in range(1,L-1,2):
        qc_tot.append(gate_zx, [i,i+1])
    for i in range(0,L-1,2):
        qc_tot.append(gate_xz, [i,i+1])
    for i in range(1,L-1,2):
        qc_tot.append(gate_xz, [i,i+1])
    for i in range(0,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    for i in range(1,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    for i in range(2,L-2,3):
        qc_tot.append(gate_zxz, [i,i+1, i+2])
    qc_tot.barrier()
    return qc_tot
############################## REVERSED TIME EVOLUTION GATES #######################################
def rev_quant_circuit(qc_tot, delta_t):
    theta_1 = J*2*delta_t
    theta_2 = hz*2*delta_t
    theta_3 = (hx/16)*2*delta_t
    theta_4 = -(3*hx/16)*2*delta_t
    theta_5 = -(3*hx/16)*2*delta_t
    theta_6 = (9*hx/16)*2*delta_t
    # 1st Evolution zz gate
    rev_gate_zz  = Hzz_gate(-theta_1)
    rev_gate_z   = Hz_gate(-theta_2)
    rev_gate_x   = Hx_gate(-theta_3)
    rev_gate_zx  = Hzx_gate(-theta_4)
    rev_gate_xz  = Hxz_gate(-theta_5)
    rev_gate_zxz = Hzxz_gate(-theta_6)
    for i in range(2,L-2,3):
        qc_tot.append(rev_gate_zxz, [i,i+1, i+2])
    for i in range(1,L-2,3):
        qc_tot.append(rev_gate_zxz, [i,i+1, i+2])
    for i in range(0,L-2,3):
        qc_tot.append(rev_gate_zxz, [i,i+1, i+2])
    for i in range(1,L-1,2):
        qc_tot.append(rev_gate_xz, [i,i+1])
    for i in range(0,L-1,2):
        qc_tot.append(rev_gate_xz, [i,i+1])
    for i in range(1,L-1,2):
        qc_tot.append(rev_gate_zx, [i,i+1])
    for i in range(0,L-1,2):
        qc_tot.append(rev_gate_zx, [i,i+1])
    for i in range(L):
        qc_tot.append(rev_gate_x, [i])
    for i in range(L):
        qc_tot.append(rev_gate_z, [i])
    qc_tot.barrier()
    for i in range(1,L,2):
        qc_tot.append(rev_gate_zz, [i,i+1])
    for i in range(0,L-1,2):
        qc_tot.append(rev_gate_zz, [i,i+1])
    qc_tot.barrier()
    return qc_tot

#############################################################################
#######################################################################
#Num_step_list = np.array([3])
#total_t = 0
#initial_t = 0.0
#final_t = 1.0

times = []
entropies = []
bitstrings_array_time = []
exp_value_array_time =[]
# -------------------------------------------------------------
# 5. Main Time Evolution Loop
# -------------------------------------------------------------
for Num_step in Num_step_list:
    total_t = 0
    delta_tV2 = 0.0
    # -------- Compute entropy at t = 0 (before evolution) -------
    times.append(delta_tV2)
    # Begining of Quantum Circuit
    qc_tot_step1 = QuantumCircuit(L)
    for i in range(L):
        qc_tot_step1.x(i)

    rev_qc_tot_step1= QuantumCircuit(L)

    # Add Quantum Circuit with qc_tot_step1
    qc_tot_step1 = quant_circuit(qc_tot_step1, delta_tV2)
    rev_qc_tot_step1 = rev_quant_circuit(rev_qc_tot_step1, delta_tV2)
    com_qc_tot_step1 = qc_tot_step1.compose(rev_qc_tot_step1)

    # target = qubit index whose entropy you measure
    qA = (L//2)-1
    qB = L//2
    qC = (L//2)+1
    bitstrings_list = []
    exp_list = []
    for P1,P2,P3 in Pauli_combination:
        if P1 == "I" and P2 == "I" and P3 == "I":
            exp = 1
            exp_list.append(exp)
            # number of shots entries, each entry is the bitstring "00"
            bits = ["000"] * shots
            bitstrings_list.append(bits)
        else:
            # REAL PART
            circ_R = measure_pauli_expectation(com_qc_tot_step1, P1, P2, P3, qA, qB, qC)
            transpiled_qc = transpile(circ_R)
            pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
            isa_circ_R = pm.run(transpiled_qc)

            job_R = sampler.run([isa_circ_R])
            counts_R = job_R.result()[0].data.c.get_counts()
            bitstrings_R = job_R.result()[0].data.c.get_bitstrings()
            bitstrings_list.append(bitstrings_R)

            exp_V2 = 0
            for bitstr, c in counts_R.items():
                b = [int(q) for q in bitstr]

                value = pauli_value_from_bits(P1, P2, P3, b)
                
                exp_V2 = exp_V2 + value *(c / shots)

            exp_list.append(exp_V2)
        
    # add bitstrings for all operator with real and Imag measurment to the bitstrings_array_time
    bitstrings_array_time.append(bitstrings_list)
    exp_value_array_time.append(exp_list)

    time_step = (final_t-initial_t)/Num_step

    qc_tot = QuantumCircuit(L)
    for i in range(L):
        qc_tot.x(i)

    rev_qc_tot= QuantumCircuit(L)
    # -------------------------------------------------------------
    # 6. Reconstruct 1-qubit reduced density matrix ρ (middle qubit)
    # -------------------------------------------------------------
    for i in range(Num_step):
        delta_t = time_step
        total_t = total_t + delta_t
        times.append(total_t)
        # Add Quantum Circuit with qc_tot
        qc_tot = quant_circuit(qc_tot, delta_t)
        rev_qc_tot = rev_quant_circuit(rev_qc_tot, delta_t)
        # Create combined circuit
        com_qc_tot = qc_tot.compose(rev_qc_tot)

        bitstrings_list = []
        exp_list = []
        for P1,P2,P3 in Pauli_combination:
            if P1 == "I" and P2 == "I" and P3 == "I":
                exp =1
                exp_list.append(exp)
                # number of shots entries, each entry is the bitstring "00"
                bits = ["000"] * shots
                bitstrings_list.append(bits)
            else:
                # REAL PART
                circ_R = measure_pauli_expectation(com_qc_tot, P1, P2, P3, qA, qB, qC)
                transpiled_qc = transpile(circ_R)
                pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
                isa_circ_R = pm.run(transpiled_qc)

                job_R = sampler.run([isa_circ_R])
                counts_R = job_R.result()[0].data.c.get_counts()
                bitstrings_R = job_R.result()[0].data.c.get_bitstrings()
                bitstrings_list.append(bitstrings_R)

                exp_V2 = 0
                for bitstr, c in counts_R.items():
                    b = [int(q) for q in bitstr]

                    value = pauli_value_from_bits(P1, P2, P3, b)
                
                    exp_V2 = exp_V2 + value *(c / shots)

                exp_list.append(exp_V2)

        # add bitstrings for all operator with real and Imag measurment to the bitstrings_array_time
        bitstrings_array_time.append(bitstrings_list)
        exp_value_array_time.append(exp_list)
    
print(f"times = {times}")
print(f"Exp_value = {np.abs(exp_value_array_time)}")
# -------------------------------------------------------------
# 7. Save results
# -------------------------------------------------------------
with open("ODR_bitstring_arrays_EE_vs_time_N101_A3_REAL_QC_2ndRun.txt", "w") as file:
    for t, bitstrings_list in zip(times, bitstrings_array_time):
        file.write(f"time={t}:\n")
        for idx,bitstring in zip(Pauli_combination, bitstrings_list):
            file.write(f" Real part of Op{idx} = {bitstring}\n")
        file.write("\n")  # blank line for readability

np.savetxt("ODR_exp_value_vs_time_hadamard_sampler_N101_A3_REAL_QC_2ndRun.txt", np.column_stack([times, np.abs(exp_value_array_time)]),
           header="time  S_A", fmt="%.6f ")
#circ_R.draw("mpl")

[('I', 'I', 'I'), ('I', 'I', 'Z'), ('I', 'Z', 'I'), ('I', 'Z', 'Z'), ('Z', 'I', 'I'), ('Z', 'I', 'Z'), ('Z', 'Z', 'I'), ('Z', 'Z', 'Z')]
times = [0.0, np.float64(0.3333333333333333), np.float64(0.6666666666666666), np.float64(1.0)]
Exp_value = [[1.     0.989  0.9825 0.9735 0.99   0.985  0.9775 0.979 ]
 [1.     0.7975 0.883  0.7835 0.8015 0.6915 0.7545 0.6365]
 [1.     0.589  0.617  0.4995 0.624  0.4915 0.592  0.5215]
 [1.     0.534  0.583  0.423  0.4815 0.3185 0.4425 0.338 ]]
